In [4]:
# Pencil, Eraser and Clear button
import cv2
import mediapipe as mp
import numpy as np
import time

# --- INITIALIZE ---
cap = cv2.VideoCapture(0)

# MediaPipe Hands: Tracks finger landmarks
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(min_detection_confidence=0.8)

# MediaPipe Selfie Segmentation: Separates the person from the background
mp_selfie = mp.solutions.selfie_segmentation
selfie = mp_selfie.SelfieSegmentation(model_selection=1) # 1 = general (heavy), 0 = landscape (fast)

# Global variables for drawing and UI state
canvas = None
prev_x, prev_y = 0, 0
current_tool = "pencil"
selected_box = None
hover_start_time = 0
CLICK_DELAY = 0.4  # Time in seconds finger must stay on a button to "click"

# Button Definitions: [x1, y1, x2, y2, tool_name, color_bgr]
buttons = [
    [50, 10, 200, 80, "pencil", (0, 255, 255)],
    [220, 10, 370, 80, "eraser", (0, 255, 255)],
    [390, 10, 540, 80, "clear", (0, 255, 255)]
]

while cap.isOpened():
    success, img = cap.read()
    if not success: break

    # Flip image horizontally for a natural mirror-like experience
    img = cv2.flip(img, 1)
    h, w, _ = img.shape
    
    # Initialize the black drawing canvas if it hasn't been created yet
    if canvas is None: 
        canvas = np.zeros_like(img)

    # 1. BLUR BACKGROUND LOGIC
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    seg_results = selfie.process(img_rgb)
    
    # The segmentation mask gives values between 0 and 1.
    # We create a boolean mask: True for person, False for background.
    mask = seg_results.segmentation_mask > 0.1
    mask_3d = np.dstack((mask, mask, mask)) # Stack to 3 channels for BGR image

    # Create a blurred version of the current frame
    blurred_img = cv2.GaussianBlur(img, (55, 55), 0)

    # Use NumPy 'where' to pick 'img' pixels for person and 'blurred_img' for background
    img = np.where(mask_3d, img, blurred_img)

    # 2. DRAW UI BUTTONS
    for x1, y1, x2, y2, name, color in buttons:
        # Fill button solid (-1) if active, else draw outline (2)
        thick = -1 if current_tool == name else 2
        cv2.rectangle(img, (x1, y1), (x2, y2), color, thick)
        
        # Invert text color based on tool selection for visibility
        text_color = (0, 0, 0) if current_tool == name else (0, 255, 255)
        cv2.putText(img, name.upper(), (x1+20, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.6, text_color, 2)

    # 3. HAND TRACKING & INTERACTION
    results = hands.process(img_rgb)
    
    if results.multi_hand_landmarks:
        lm = results.multi_hand_landmarks[0].landmark
        
        # Landmark 8 is the Index Finger Tip
        cx, cy = int(lm[8].x * w), int(lm[8].y * h)
        
        # Check if index finger is extended (Tip Y is higher than Pip Joint Y)
        index_up = lm[8].y < lm[6].y 

        # A. BUTTON INTERACTION (Hover to click logic)
        in_button = False
        for x1, y1, x2, y2, name, color in buttons:
            if x1 < cx < x2 and y1 < cy < y2:
                in_button = True
                if selected_box != name:
                    selected_box = name
                    hover_start_time = time.time() # Start selection timer

                # Calculate how long the finger has hovered
                elapsed = time.time() - hover_start_time
                progress = min(elapsed / CLICK_DELAY, 1.0)
                
                # Visual Feedback: Green ring and white progress circle around fingertip
                cv2.circle(img, (cx, cy), 15, (0, 255, 0), 2)
                cv2.ellipse(img, (cx, cy), (15, 15), 0, 0, progress*360, (255, 255, 255), 3)

                # If hover time exceeds delay, change the tool
                if elapsed >= CLICK_DELAY:
                    current_tool = name
                    if name == "clear": 
                        canvas = np.zeros_like(img) # Wipe the drawing layer
        
        if not in_button: selected_box = None

        # B. DRAWING LOGIC (Only active below the UI header)
        if index_up and cy > 100:
            if prev_x == 0: prev_x, prev_y = cx, cy # Initial touch point
            
            if current_tool == "pencil":
                cv2.line(canvas, (prev_x, prev_y), (cx, cy), (0, 255, 255), 5)
            elif current_tool == "eraser":
                # Erase by drawing black lines on the black canvas
                cv2.line(canvas, (prev_x, prev_y), (cx, cy), (0, 0, 0), 50)
            
            prev_x, prev_y = cx, cy # Update tracking coordinates
        else:
            prev_x, prev_y = 0, 0 # Reset when finger is released or in UI area

        # Draw a small dot at the cursor point
        cv2.circle(img, (cx, cy), 5, (255, 255, 255), -1)

    # 4. MERGE CANVAS & DISPLAY
    # Convert canvas to grayscale to create a binary mask
    canvas_gray = cv2.cvtColor(canvas, cv2.COLOR_BGR2GRAY)
    _, drawing_mask = cv2.threshold(canvas_gray, 1, 255, cv2.THRESH_BINARY)
    
    # Overwrite camera frame pixels where the canvas mask is active
    img[drawing_mask > 0] = canvas[drawing_mask > 0]

    cv2.imshow("Air Draw Blurred", img)
    if cv2.waitKey(1) & 0xFF == 27: break # Exit on ESC key

cap.release()
cv2.destroyAllWindows()

I0000 00:00:1773756158.096154  128952 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1773756158.096916  131217 gl_context.cc:344] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: Mesa Intel(R) Xe Graphics (TGL GT2)
I0000 00:00:1773756158.098699  128952 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1773756158.099683  131228 gl_context.cc:344] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: Mesa Intel(R) Xe Graphics (TGL GT2)
